In [ ]:

import torch
from torch import nn
from torch.nn import functional as F
from torch.autograd import Function
from torch.nn.parameter import Parameter
from torch import optim

#from tqdm import tqdm
from IPython import display
import time
import pickle 
import matplotlib.pyplot as plt
import random
import os

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [ ]:
import numpy as np

def load_point_cloud_from_obj(filepath):
    vertices = []

    with open(filepath, 'r') as file:
        for line in file:
            if line.startswith('v '):  # only vertex positions
                parts = line.strip().split()
                vertex = list(map(float, parts[1:4]))
                vertices.append(vertex)

    return np.array(vertices, dtype=np.float32)

# Load the provided .obj file
file_path = "D:\PYTHON_code\HCP-main\simu_flow\elephant-reference.obj"
point_cloud1 = load_point_cloud_from_obj(file_path)
point_cloud1.shape
indices = np.random.choice(point_cloud1.shape[0], 200, replace=False)
elepant = point_cloud1[indices]

point_cloud2 = load_point_cloud_from_obj('D:\PYTHON_code\HCP-main\simu_flow\horse-01.obj')
point_cloud2.shape
indices = np.random.choice(point_cloud2.shape[0], 200, replace=False)
horse = point_cloud2[indices]

point_cloud3 = load_point_cloud_from_obj('D:/PYTHON_code/HCP-main/simu_flow/flam-reference.obj')
point_cloud3.shape
indices = np.random.choice(point_cloud3.shape[0], 200, replace=False)
flam = point_cloud3[indices]

In [ ]:
figure = [elepant,horse,flam]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import imageio
import os
step_indices = [0, 20, 40, 60, 80, 100]
colors = ['#2C3E50', '#A2B9C8', '#36597D', '#D6E4F0',
'#E37C59', '#F5BCA9', '#C95D39', '#FFDAC6',
'#4A7C59', '#B4CFB0', '#6F9E84', '#D5E7D2']
# 创建 5 行的子图布局，前3行为 1x6，第4-5行为合并1行，放3个图
fig = plt.figure(figsize=(22, 10))
# 替换原来的 gs = fig.add_gridspec(5, 6)
gs_top = fig.add_gridspec(nrows=3, ncols=6, top=0.95, bottom=0.45, hspace=0.05, wspace=0.05)
gs_bottom = fig.add_gridspec(nrows=2, ncols=6, top=0.4, bottom=0.11, hspace=0.4, wspace=0.5)

#gs = fig.add_gridspec(5, 6)  # 5 行 6 列
for t in range(3):
    X = figure[t]
    steps = np.linspace(0, 1, 100)
    Z = []

    # 创建临时保存图片的文件夹
    os.makedirs("frames", exist_ok=True)
    filenames = []
    loss3 = []
    loss4 = []
    name = ['Elepant','Horse','Flam']
    angle = [50,60,70]
    noise_factor = np.random.normal(scale=0, size=X.shape)
    steps = np.linspace(0, 1.1, 101) 
    for idx, i in enumerate(steps):
        # 信号功率
        signal_power = np.sum(X**2)

        # 目标最终 SNR（单位：dB）
        target_snr_db = 30

        # 计算目标总噪声功率
        target_noise_power = signal_power / (10**(target_snr_db / 10))
        noise_factor_new = np.random.normal(scale=np.sqrt(target_noise_power/(101* X.size)), size=X.shape) 
        noise_factor  =  noise_factor + noise_factor_new
        zi = X + noise_factor  # add noise to 
        Z.append(zi)
        loss3.append(RIOT(zi,X,maxed=False, rep = 50,sliced = True,rd_rad = 3)[1])
        loss4.append(RIOT(zi,X,maxed=True, rep = 50,sliced = True,rd_rad = 3)[1])



    for j, idx0 in enumerate(step_indices):
        if (j==0):
                ax = fig.add_subplot(gs_top[t, j], projection='3d')
                zi = Z[idx0]  # 第 t 个 figure 的第 idx 步
                ax.scatter(zi[:, 2], zi[:, 0], zi[:, 1], c=colors[4*t+1], s=0.5,alpha=0.8)
                ax.set_title(f'Source shape : {name[t]}', fontsize=22, fontname='Times New Roman',fontweight='bold')
                ax.view_init(elev=0, azim=angle[t])
                ax.set_axis_off()
        else:
                ax = fig.add_subplot(gs_top[t, j], projection='3d')
                zi = Z[idx0]  # 第 t 个 figure 的第 idx 步
                ax.scatter(zi[:, 2], zi[:, 0], zi[:, 1], c=colors[4*t], s=0.5,alpha=0.8)
                snr_values = round(10*np.log10(100*10**(target_snr_db/10)) - 10 * np.log10(idx0))
                ax.set_title(f'Step {idx0}: SNR = {snr_values}', fontsize=22, fontname='Times New Roman')
                ax.view_init(elev=0, azim=angle[t])
                ax.set_axis_off()

for t in range(3):
        ax = fig.add_subplot(gs_bottom[0:, 2*t:2*t+2])  # 每个图占两列宽
        ax.plot(loss3, 
                color=colors[4*t+2], 
                linestyle='--', 
                linewidth=1, 
                marker='o', 
                markersize=3, 
                label='Expected-RIOT')

        ax.plot(loss4, 
                color=colors[4*t+3], 
                linestyle='-', 
                linewidth=1, 
                marker='x', 
                markersize=3, 
                label='Maxed-RIOT')
        ax.set_title(f'Loss Curve for {name[t]}',fontsize=22,fontname='Times New Roman')
        ax.set_xlabel('Step',fontsize=22,fontname='Times New Roman')
        ax.set_ylabel('Loss',fontsize=22,fontname='Times New Roman')
        ax.tick_params(axis='both', labelsize=20)
        ax.legend(fontsize=15, frameon=False, loc='lower right')
#plt.tight_layout(pad=0.8)
#fig.subplots_adjust(wspace=0.15, hspace=0.25)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 创建 reference 数量
Rep = np.arange(1, 1000, 5)
num_repeats = 20

# 初始化结果数组
results1 = np.zeros((num_repeats, len(Rep)))
results2 = np.zeros((num_repeats, len(Rep)))

# 进行10次重复实验
for i in range(num_repeats):
    for j, rp in enumerate(Rep):
        results1[i, j] = RIOT(figure[1], figure[2], maxed=False, rep=rp, sliced=True, rd_rad=2, determin=False)[1]
        results2[i, j] = RIOT(figure[1], figure[2], maxed=True,  rep=rp, sliced=True, rd_rad=2, determin=False)[1]

# 计算均值和标准差
mean1 = np.mean(results1, axis=0)
std1 = np.std(results1, axis=0)
mean2 = np.mean(results2, axis=0)
std2 = np.std(results2, axis=0)

# 绘图
plt.figure(figsize=(15, 6.5))

# Expected-RIOT 曲线与阴影
plt.plot(Rep, mean1, linestyle='--', marker='s', linewidth=2, markersize=1,
         color='#ff7f0e', label='Expected-DiMOT')
plt.fill_between(Rep, mean1 - std1, mean1 + std1, color='#ff7f0e', alpha=0.3)

# Maxed-RIOT 曲线与阴影
plt.plot(Rep, mean2, linestyle='-', marker='o', linewidth=2, markersize=1,
         color='#1f77b4', label='Maxed-DiMOT')
plt.fill_between(Rep, mean2 - std2, mean2 + std2, color='#1f77b4', alpha=0.3)

# 美化图表
plt.xlabel("Number of reference distributions", fontsize=20, fontname='Times New Roman')
plt.ylabel("Loss", fontsize=20, fontname='Times New Roman')
plt.legend(fontsize=18,loc='upper left')
plt.xticks(fontsize=20, fontname='Times New Roman')
plt.yticks(fontsize=20, fontname='Times New Roman')
plt.tight_layout()
plt.show()